# Deep Learning 083 — The Transformer Decoder (Training)

Companion notebook to the lesson. Every part now exists, so this notebook assembles them —
and then spends most of its time on the two things that are easy to get wrong and hard to
notice.

| Step | What we measure |
|---|---|
| shapes | 4 × 512 at every stage; changes exactly **once**, at the output |
| the bill | one decoder block **4,204,032**, asserted against the module |
| the output block | **5,130,000** at V = 10,000 — tying saves all of it |
| the sanity check | untrained loss = **ln(V)**: 9.31 measured vs 9.21 predicted |
| **the right-shift bug** | trained on **pure noise**: 100% accuracy, loss 0.0016 |
| done correctly | ~2.6% — which is chance |
| one pass, n losses | identical to computing them one prefix at a time |

Needs `torch` (CPU is fine). Part D trains briefly.

In [ ]:
import math, time
import torch
import torch.nn as nn

D, D_FF, H = 512, 2048, 8

def causal_mask(n):
    return torch.full((n, n), float("-inf")).triu(1)

class MHA(nn.Module):
    '''One class for both flavours: pass the same tensor twice for self-attention,
       or (decoder, encoder) for cross attention.'''
    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk = nn.Linear(d, d), nn.Linear(d, d)
        self.wv, self.wo = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, q_src, kv_src, masked=False):
        nq, nk = len(q_src), len(kv_src)
        q = self.wq(q_src).view(nq, self.h, self.dk).transpose(0, 1)
        k = self.wk(kv_src).view(nk, self.h, self.dk).transpose(0, 1)
        v = self.wv(kv_src).view(nk, self.h, self.dk).transpose(0, 1)
        s = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if masked:
            s = s + causal_mask(nq)
        w = torch.softmax(s, dim=-1)
        return self.wo((w @ v).transpose(0, 1).reshape(nq, self.h * self.dk))

class DecoderBlock(nn.Module):
    '''THREE sub-blocks, not two.'''
    def __init__(self, d=D, d_ff=D_FF):
        super().__init__()
        self.self_attn, self.cross_attn = MHA(d), MHA(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, enc, masked=True):
        x = self.ln1(x + self.self_attn(x, x, masked=masked))   # lesson 081
        x = self.ln2(x + self.cross_attn(x, enc))               # lesson 082
        x = self.ln3(x + self.ff(x))                            # lesson 080
        return x

That is the whole decoder block. Compare it with the encoder block from lesson 080: the only
structural difference is the middle line.

## Part A — The shape survives, and the bill

Derive the parameter count on paper first, then let the module check you.

In [ ]:
torch.manual_seed(1083)
n_tgt, n_src, V = 4, 3, 10_000        # "START हम दोस्त हैं"  /  "We are friends"
enc = torch.randn(n_src, D)
x = torch.randn(n_tgt, D)
blk = DecoderBlock()

a = blk.ln1(x + blk.self_attn(x, x, masked=True));  print("masked self-attn + A&N ", tuple(a.shape))
c = blk.ln2(a + blk.cross_attn(a, enc));            print("cross attention  + A&N ", tuple(c.shape))
f = blk.ln3(c + blk.ff(c));                         print("feed-forward     + A&N ", tuple(f.shape))

y = x
for _ in range(6):
    y = blk(y, enc)
print("after all six blocks   ", tuple(y.shape))

head = nn.Linear(D, V)
logits = head(y)
print("output block           ", tuple(logits.shape))
print("row sums after softmax :", torch.softmax(logits, -1).sum(-1))
assert tuple(f.shape) == (n_tgt, D)

512 the whole way, and the dimension changes **exactly once** — at the final projection.

Look closely at the cross-attention line: **4 queries against 3 encoder vectors, and 4 come
out.** The source length disappears inside the block. That is why a 3-word English sentence
can become a 4-token Hindi one, and why the *decoder* controls the output length.

In [ ]:
attn = 4 * (D*D + D)
ff   = (D*D_FF + D_FF) + (D_FF*D + D)
ln   = 2*D
block = 2*attn + ff + 3*ln
assert sum(p.numel() for p in DecoderBlock().parameters()) == block

print(f"masked self-attention : {attn:>11,}")
print(f"cross attention       : {attn:>11,}")
print(f"feed-forward network  : {ff:>11,}")
print(f"three layer norms     : {3*ln:>11,}")
print(f"ONE DECODER BLOCK     : {block:>11,}   (module agrees)")
print(f"six blocks            : {6*block:>11,}")
print(f"the output block      : {D*V + V:>11,}   <- at V = {V:,}")

## Part B — The output block is a second embedding table

The embedding maps V words to 512 numbers. The output block maps 512 numbers back to V
scores. Same two shapes, one transposed relative to the other.

In [ ]:
stack = 6 * block
print(f"{'V':>8} {'embedding':>13} {'output block':>14} {'tying saves':>13} {'of model':>10}")
for V_ in (5_000, 10_000, 37_000, 50_000):
    emb, out = V_*D, V_*D + V_
    total = stack + emb + out
    print(f"{V_:>8,} {emb:>13,} {out:>14,} {emb:>13,} {100*emb/total:>9.1f}%")

At V = 50,000 the two tables together outweigh all six decoder blocks, and tying deletes one
outright — **a third of the model**. It is not free: it asserts that the vector used to look a
word **up** is the vector used to score it on the way **out**. That happens to help in
practice (Press & Wolf, 2017), but it is an architectural claim rather than a pure saving.

## Part C — The sanity check to memorise

At initialisation the softmax is near-uniform, so the cross-entropy of the correct word is
$-\ln(1/V) = \ln V$. **Predict the three numbers before running this.**

In [ ]:
torch.manual_seed(3083)
print(f"{'V':>8} {'ln(V)':>9} {'measured':>10} {'max prob':>10} {'uniform 1/V':>13}")
for V_ in (1_000, 10_000, 50_000):
    h = nn.Linear(D, V_)
    logits = h(torch.randn(64, D))
    loss = nn.functional.cross_entropy(logits, torch.randint(0, V_, (64,))).item()
    p = torch.softmax(logits, -1)
    print(f"{V_:>8,} {math.log(V_):>9.4f} {loss:>10.4f} {p.max():>10.5f} {1/V_:>13.6f}")

**A first-step loss of 9.21 on a 10,000-word vocabulary is not a problem — it is the model
correctly knowing nothing.** Much higher means bad initialisation. Much *lower* means the
answer is leaking in, which is Part D.

## Part D — Right shifting, and what forgetting it does

The decoder input is the target with START prepended, so position $i$ holds token $i-1$ and
must predict token $i$. Drop the shift and position $i$ holds token $i$ and must predict
token $i$ — which it can already see on its own input line.

The targets below are drawn **uniformly at random**, so the data contains no signal at all.
Correct behaviour is chance. Predict both rows before running it.

In [ ]:
V_, N, d = 50, 8, 64
START = 0

def train_and_score(shifted, steps=400):
    torch.manual_seed(4083)
    emb = nn.Embedding(V_, d)
    pos = nn.Parameter(torch.randn(N + 1, d) * 0.02)
    attn = MHA(d, h=4)
    ln = nn.LayerNorm(d)
    head = nn.Linear(d, V_)
    params = list(emb.parameters()) + [pos] + list(attn.parameters()) + \
             list(ln.parameters()) + list(head.parameters())
    opt = torch.optim.Adam(params, lr=3e-3)

    def feed(tgt_row):
        return torch.cat([torch.tensor([START]), tgt_row[:-1]]) if shifted else tgt_row

    for _ in range(steps):
        tgt = torch.randint(1, V_, (16, N))
        loss = 0.0
        for b in range(len(tgt)):
            inp = feed(tgt[b])
            x = emb(inp) + pos[:len(inp)]
            h = ln(x + attn(x, x, masked=True))
            loss = loss + nn.functional.cross_entropy(head(h), tgt[b])
        loss = loss / len(tgt)
        opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        tgt = torch.randint(1, V_, (200, N))
        acc = 0.0
        for b in range(len(tgt)):
            inp = feed(tgt[b])
            x = emb(inp) + pos[:len(inp)]
            h = ln(x + attn(x, x, masked=True))
            acc += (head(h).argmax(-1) == tgt[b]).float().mean().item()
    return loss.item(), acc / len(tgt)

for shifted in (False, True):
    loss, acc = train_and_score(shifted)
    tag = "right-shifted (correct)" if shifted else "NOT shifted (the bug)"
    print(f"{tag:>26}:  loss {loss:>7.4f}   accuracy {100*acc:>6.2f}%")

print(f"\nchance on this data would be {100/(V_-1):.1f}%, with loss near ln({V_-1}) = {math.log(V_-1):.2f}")

**Without the shift the model reaches ~100% on data that contains nothing to learn**, because
the answer is sitting on its own input line. With the shift it correctly fails.

This is the most common way a decoder is silently wrong, and the dangerous part is that
**the failure mode is that everything looks wonderful** — the loss curve is beautiful. The
check is not "is the loss falling" but "is the loss falling below what this data could
possibly support".

## Part E — One pass, n loss terms

In [ ]:
torch.manual_seed(5083)
n_tgt, V_ = 6, 500
enc = torch.randn(4, D)
x = torch.randn(n_tgt, D)
blk = DecoderBlock()
head = nn.Linear(D, V_)
target = torch.randint(0, V_, (n_tgt,))

with torch.no_grad():
    par = nn.functional.cross_entropy(head(blk(x, enc)), target, reduction="none")
    seq = torch.stack([
        nn.functional.cross_entropy(head(blk(x[:t+1], enc))[-1:], target[t:t+1])
        for t in range(n_tgt)
    ])

print("one pass over the whole target :", "  ".join(f"{v:.4f}" for v in par))
print("one prefix at a time           :", "  ".join(f"{v:.4f}" for v in seq))
print(f"max difference                 : {(par-seq).abs().max():.2e}")

Six positions, six loss terms, **one forward pass**. An $n$-word sentence is a single training
example that supplies $n$ supervised predictions — which is the real reason language models
are cheap to train relative to how much they learn, and it is available only because of the
mask (lesson 081).

## What to take away

- **Input block:** right-shift with START, tokenise, embed, add positional encoding.
- **Three sub-blocks per decoder block**, not two. Shape stays $n \times 512$ and changes
  exactly once, at the output.
- **One decoder block is 4,204,032 parameters**; the output block alone is 5,130,000 at
  V = 10,000, and **tying it to the embedding saves a third of the model at V = 50,000**.
- **Untrained loss is ln(V).** Memorise it; it catches most wiring bugs on step one.
- **The right shift is what makes the task well-posed.** Forget it and the model scores 100%
  on noise.

**Exercises**

1. Implement weight tying (`head.weight = emb.weight`) in Part D and confirm the parameter
   count drops by exactly `V*d`. Does the model still train?
2. In Part C, replace the random final vectors with the output of an actual decoder block.
   Is the loss still ln(V)? Should it be?
3. Re-run Part D with the shift but **without** the causal mask. What accuracy do you get,
   and why?
4. Add an EOS token to the targets in Part D and check that the model learns to emit it at
   the right position.